# Unix Shell Tutorial: Semantics and Ontologies

This is the **eighth and last tutorial** in a series that will demonstrate how shell scripting can be used to perform the tasks that health and life science specialists may need to undertake to find and retrieve biomedical data and text. We will use the compound caffeine as an example and explore different public repositories to identify diseases related to it. The focus is not on the specific relationships we may discover, but on the process of obtaining them.

The objective of this tutorial is to introduce the world of semantics, and present step-by-step examples to enhance text and data processing by using semantics. The goal is to equip the reader with the basic set of skills to explore semantic resources that are nowadays available using simple shell script commands.

> This tutorial is part of a series of tutorials adapted as interactive versions of the hands-on steps described in the [Data and Text Processing for Health and Life Sciences](https://labs.rd.ciencias.ulisboa.pt/book/) book, which is licensed under the [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

## Classes

In the previous tutorials we searched for mentions of caffeine and malignant hyperthermia in text. However, we may miss related entities that may also be of our interest. These related entities can be found in semantic resources, such as ontologies. The semantics of caffeine and malignant hyperthermia are represented in ChEBI and DO ontologies, respectively.

### OWL files

The original OWL files can be retrieved by using `curl`:


In [1]:
%%bash
curl -s -L -O "http://purl.obolibrary.org/obo/doid/releases/2021-03-29/doid.owl"
curl -s -L -O "http://purl.obolibrary.org/obo/chebi/198/chebi_lite.owl"
du -h *.owl

159M	chebi_lite.owl
27M	doid.owl


The `-O` option saves the content to a local file named according to the name of the remote file, usually the last part of the URL. The equivalent long form to the `-O` option is `--remote-name`. The option `-L` enables the curl command to follow a [URL redirection](https://en.wikipedia.org/wiki/URL_redirection). The equivalent long form to the `-L` option is `--location`.

The previous `curl` commands will create the files `chebi_lite.owl` and `doid.owl`, respectively. Note that these links are for the specific releases used in the book, and using another release may change the output of the examples presented in it. To retrieve the most recent release we should use the following links:

- http://purl.obolibrary.org/obo/doid.owl
- http://purl.obolibrary.org/obo/chebi/chebi_lite.owl

To find other ontology links search for them on the [BioPortal](http://bioportal.bioontology.org/) or on the [OBO Foundry](http://www.obofoundry.org/) webpages. Alternatively, we can also get the OWL files from the [book file archive](http://labs.rd.ciencias.ulisboa.pt/book/).

Since the original files are very large OWL files containing numerous classes, we have reduced the number of classes to avoid long waiting times for command line execution.

> Using the reduced OWL files preloaded in this tutorial, the command lines described here may work in a similar, but not identical, manner.
The following command downloads the reduced files directly from the GitHub repository:


In [2]:
%%bash
curl -s -O 'https://raw.githubusercontent.com/lasigeBioTM/data-text-processing-notebooks/refs/heads/main/data/chebi_lite.owl'
curl -s -O 'https://raw.githubusercontent.com/lasigeBioTM/data-text-processing-notebooks/refs/heads/main/data/doid.owl'
du -h *.owl

56K	chebi_lite.owl
144K	doid.owl


File sizes for `doid.owl` and `chebi_lite.owl` (approx a few KB each). As you can see, these files are only a few KB in size, while the original ones are several MB.


### Class label

Both OWL files use the XML format syntax. Thus, to check if our entities are represented in the ontology, we can search for ontology elements that contain them using a simple grep command:

In [3]:
%%bash
grep '>malignant hyperthermia<' doid.owl

        <rdfs:label rdf:datatype="http://www.w3.org/2001/XMLSchema#string">malignant hyperthermia</rdfs:label>


**Expected Output:**
Lines containing the label "malignant hyperthermia".

In [4]:
%%bash
grep '>caffeine<' chebi_lite.owl

        <rdfs:label rdf:datatype="http://www.w3.org/2001/XMLSchema#string">caffeine</rdfs:label>


**Expected Output:**
Lines containing the label "caffeine".

For each `grep` the output will be the line that describes the property label (`rdfs:label`), which is inside the definition of the class that represents the entity.

### Class definition

To retrieve the full class definition, a more efficient approach is to use the `xmllint` command, which we already used in previous tutorials to process XML.
To install `xmllint` we can execute:

In [5]:
%%bash
apt-get update && apt-get install -y libxml2-utils

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.0 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,751 kB]
Get:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,728 kB]
Get:14 h

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


 Now we can execute the `xmllint` command:


In [6]:
%%bash
xmllint --xpath "//*[local-name()='label' and text()='malignant hyperthermia']/.." doid.owl

<owl:Class rdf:about="http://purl.obolibrary.org/obo/DOID_8545">
        <rdfs:subClassOf rdf:resource="http://purl.obolibrary.org/obo/DOID_0050736"/>
        <rdfs:subClassOf rdf:resource="http://purl.obolibrary.org/obo/DOID_66"/>
        <rdfs:subClassOf>
            <owl:Restriction>
                <owl:onProperty rdf:resource="http://purl.obolibrary.org/obo/IDO_0000664"/>
                <owl:someValuesFrom rdf:resource="http://purl.obolibrary.org/obo/GENO_0000147"/>
            </owl:Restriction>
        </rdfs:subClassOf>
        <obo:IAO_0000115 rdf:datatype="http://www.w3.org/2001/XMLSchema#string">A muscle tissue disease that is characterized by a drastic and uncontrolled increase in skeletal muscle oxidative metabolism, which overwhelms the body's capacity to supply oxygen, remove carbon dioxide, and regulate body temperature.</obo:IAO_0000115>
        <oboInOwl:hasDbXref rdf:datatype="http://www.w3.org/2001/XMLSchema#string">GARD:6964</oboInOwl:hasDbXref>
        <oboInOwl:

**Expected Output:**
The full XML block for the malignant hyperthermia class.

The XPath query starts by finding the label that contains malignant hyperthermia and then `..` gives the parent element, in this case the Class element. From the output we can see that the semantics of malignant hyperthermia is much more than its label.

For example, we can check that malignant hyperthermia is a subclass of (specialization) the entries [0050736](http://purl.obolibrary.org/obo/DOID_0050735) and [66](http://purl.obolibrary.org/obo/DOID_66). By clicking in the previous links will see that malignant hyperthermia is a special case of a autosomal dominant disease and of a muscle tissue disease.

We can search for those specific relations between malignant hyperthermia and the entries `0050736` and `66`:

In [7]:
%%bash
xmllint --xpath "//*[local-name()='label' and text()='malignant hyperthermia']/..//*[@*[local-name()='resource' and .='http://purl.obolibrary.org/obo/DOID_66' or .='http://purl.obolibrary.org/obo/DOID_0050736']]" doid.owl

<rdfs:subClassOf rdf:resource="http://purl.obolibrary.org/obo/DOID_0050736"/>
<rdfs:subClassOf rdf:resource="http://purl.obolibrary.org/obo/DOID_66"/>


**Expected Output:**
The XML lines describing the subclass relations.

We added the `@*[local-name()='resource']` to extract the URI specified in an attribute resource of any descendant element `//*[...]`.
The relation specification uses the `subClassOf` element.

We can do the same to retrieve the full class definition of caffeine:

In [8]:
%%bash
xmllint --xpath "//*[local-name()='label' and text()='caffeine']/.." chebi_lite.owl

<owl:Class rdf:about="http://purl.obolibrary.org/obo/CHEBI_27732">
        <rdfs:subClassOf rdf:resource="http://purl.obolibrary.org/obo/CHEBI_26385"/>
        <rdfs:subClassOf rdf:resource="http://purl.obolibrary.org/obo/CHEBI_27134"/>
        <rdfs:subClassOf>
            <owl:Restriction>
                <owl:onProperty rdf:resource="http://purl.obolibrary.org/obo/RO_0000087"/>
                <owl:someValuesFrom rdf:resource="http://purl.obolibrary.org/obo/CHEBI_25435"/>
            </owl:Restriction>
        </rdfs:subClassOf>
        <rdfs:subClassOf>
            <owl:Restriction>
                <owl:onProperty rdf:resource="http://purl.obolibrary.org/obo/RO_0000087"/>
                <owl:someValuesFrom rdf:resource="http://purl.obolibrary.org/obo/CHEBI_35337"/>
            </owl:Restriction>
        </rdfs:subClassOf>
        <rdfs:subClassOf>
            <owl:Restriction>
                <owl:onProperty rdf:resource="http://purl.obolibrary.org/obo/RO_0000087"/>
              

**Expected Output:**
The full XML block for the caffeine class.

From the output we can see that the types of semantics available for caffeine differs from the semantics of malignant hyperthermia, but they still share many important properties, such as the definition of `subClassOf`.

The class caffeine is a specialization of two other entries: [26385](http://purl.obolibrary.org/obo/CHEBI_26385) (purine alkaloid), and [27134](http://purl.obolibrary.org/obo/CHEBI_26385) (trimethylxanthine).
We can search for those specific relations between caffeine and the entries `26385` and `27134`:

In [9]:
%%bash
xmllint --xpath "//*[local-name()='label' and text()='caffeine']/..//*[@*[local-name()='resource' and .='http://purl.obolibrary.org/obo/CHEBI_26385' or .='http://purl.obolibrary.org/obo/CHEBI_27134']]" chebi_lite.owl

<rdfs:subClassOf rdf:resource="http://purl.obolibrary.org/obo/CHEBI_26385"/>
<rdfs:subClassOf rdf:resource="http://purl.obolibrary.org/obo/CHEBI_27134"/>


**Expected Output:**
The XML lines describing the subclass relations for caffeine.

The relation specification uses the `subClassOf` element.

### Related Classes

There are additional subclass relationships that do not represent subsumption (*is-a*).
For example, the relationship between caffeine and the entry [25435](http://purl.obolibrary.org/obo/CHEBI_25435) (mutagen) is defined by the entry [0000087](http://purl.obolibrary.org/obo/RO_0000087) (has role) of the Relations Ontology.
This means that the relationship defines that caffeine has role mutagen.
We can search that specific relation between caffeine and mutagen (CHEBI:25435):

In [10]:
%%bash
xmllint --xpath "//*[local-name()='label' and text()='caffeine']/..//*[@*[local-name()='resource' and .='http://purl.obolibrary.org/obo/CHEBI_25435']]/../.." chebi_lite.owl

<rdfs:subClassOf>
            <owl:Restriction>
                <owl:onProperty rdf:resource="http://purl.obolibrary.org/obo/RO_0000087"/>
                <owl:someValuesFrom rdf:resource="http://purl.obolibrary.org/obo/CHEBI_25435"/>
            </owl:Restriction>
        </rdfs:subClassOf>


**Expected Output:**
The XML block defining the "has role" relation.

The specification uses the Restriction element.

We can now search in the OWL file for the definition of the type of relation has role (`RO:0000087`):

In [11]:
%%bash
xmllint --xpath "//*[local-name()='ObjectProperty'][@*[local-name()='about']='http://purl.obolibrary.org/obo/RO_0000087']" chebi_lite.owl

<owl:ObjectProperty rdf:about="http://purl.obolibrary.org/obo/RO_0000087">
        <oboInOwl:hasDbXref rdf:datatype="http://www.w3.org/2001/XMLSchema#string">RO:0000087</oboInOwl:hasDbXref>
        <oboInOwl:hasOBONamespace rdf:datatype="http://www.w3.org/2001/XMLSchema#string">chebi_ontology</oboInOwl:hasOBONamespace>
        <oboInOwl:id rdf:datatype="http://www.w3.org/2001/XMLSchema#string">has_role</oboInOwl:id>
        <oboInOwl:is_cyclic rdf:datatype="http://www.w3.org/2001/XMLSchema#boolean">false</oboInOwl:is_cyclic>
        <oboInOwl:is_transitive rdf:datatype="http://www.w3.org/2001/XMLSchema#boolean">false</oboInOwl:is_transitive>
        <oboInOwl:shorthand rdf:datatype="http://www.w3.org/2001/XMLSchema#string">has_role</oboInOwl:shorthand>
        <rdfs:label rdf:datatype="http://www.w3.org/2001/XMLSchema#string">has role</rdfs:label>
    </owl:ObjectProperty>


**Expected Output:**
The ObjectProperty definition for RO:0000087.

The XPath query starts by finding the elements ObjectProperty and then selects the ones containing the about attribute with the relation URI as value.
We can check that the relation is neither transitive nor cyclic.

## URIs and Labels

In the previous examples, we searched the OWL file using labels and URIs. To standardize the process, we will create two scripts that will convert a label into a URI and vice-versa. The idea is to perform all the internal ontology processing using the URIs and in the end convert them to labels, so we can use them in text processing.

#### URI of a label

To get the URI of malignant hyperthermia, we can use the following query:

In [12]:
%%bash
xmllint --xpath "//*[local-name()='label' and text()='malignant hyperthermia']/../@*[local-name()='about']" doid.owl

 rdf:about="http://purl.obolibrary.org/obo/DOID_8545"


**Expected Output:**
`rdf:about="http://purl.obolibrary.org/obo/DOID_8545"`

We added the `@*[local-name()='about']` to extract the URI specified as an attribute of that class.
The output will be the name of the attribute and its value:

```xml
rdf:about="http://purl.obolibrary.org/obo/DOID_8545"
```

To extract only the value, we can add the string function to the XPath query:

In [13]:
%%bash
xmllint --xpath "string(//*[local-name()='label' and text()='malignant hyperthermia']/../@*[local-name()='about'])" doid.owl

http://purl.obolibrary.org/obo/DOID_8545


**Expected Output:**
`http://purl.obolibrary.org/obo/DOID_8545`

The output will now be only the attribute value.
Unfortunately, the string function returns only one attribute value, even if many are matched. Nonetheless, we use the string function because we assume that malignant hyperthermia is an unambiguous label, i.e. only one class will match. To avoid this limitation, we can use the `cut` command to extract only the URI value, using the double-quote character (`"`) as the delimiter:

In [14]:
%%bash
xmllint --xpath "//*[local-name()='label' and text()='malignant hyperthermia']/../@*[local-name()='about']" doid.owl | cut -d'"' -f2

http://purl.obolibrary.org/obo/DOID_8545


**Expected Output:**
`http://purl.obolibrary.org/obo/DOID_8545`

The same command applies to caffeine:

In [15]:
%%bash
xmllint --xpath "//*[local-name()='label' and text()='caffeine']/../@*[local-name()='about']" chebi_lite.owl | cut -d'"' -f2

http://purl.obolibrary.org/obo/CHEBI_27732


**Expected Output:**
`http://purl.obolibrary.org/obo/CHEBI_27732`

We can now write a script that receives multiple labels given as standard input and the OWL file where to find the URIs as argument. Thus, we can create the script named `geturi.sh`:

In [16]:
%%bash
cat << 'EOF' > geturi.sh
OWLFILE=$1
xargs -I {} xmllint --xpath "//*[local-name()='label' and text()='{}']/../@*[local-name()='about']" $OWLFILE | cut -d'"' -f2
EOF

Again we cannot forget to save the file in our working directory, and add the right permissions using `chmod` as we did with our scripts in the previous tutorials.

In [17]:
%%bash
chmod u+x geturi.sh

**Expected Output:**
No output.

The `xargs` command is used to process each line of the standard input.
Now to execute the script we only need to provide the labels as standard input:

In [18]:
%%bash
echo 'malignant hyperthermia' | ./geturi.sh doid.owl

http://purl.obolibrary.org/obo/DOID_8545


**Expected Output:**
`http://purl.obolibrary.org/obo/DOID_8545`

In [19]:
%%bash
echo 'caffeine' | ./geturi.sh chebi_lite.owl

http://purl.obolibrary.org/obo/CHEBI_27732


**Expected Output:**
`http://purl.obolibrary.org/obo/CHEBI_27732`

The output should be the URIs of those classes.

We can also execute the script using multiple labels, one per line:

In [20]:
%%bash
echo -e 'malignant hyperthermia\nmuscle tissue disease' | ./geturi.sh doid.owl

http://purl.obolibrary.org/obo/DOID_8545
http://purl.obolibrary.org/obo/DOID_66


**Expected Output:**
`http://purl.obolibrary.org/obo/DOID_8545`
`http://purl.obolibrary.org/obo/DOID_66`

In [21]:
%%bash
echo -e 'caffeine\npurine alkaloid\ntrimethylxanthine' | ./geturi.sh chebi_lite.owl

http://purl.obolibrary.org/obo/CHEBI_27732
http://purl.obolibrary.org/obo/CHEBI_26385
http://purl.obolibrary.org/obo/CHEBI_27134


**Expected Output:**
URIs for caffeine, purine alkaloid, and trimethylxanthine.

The output will be a URI for each label.

#### Label of a URI

To get the label of the disease entry with the identifier `8545`, we can also use the `xmllint` command:

In [22]:
%%bash
xmllint --xpath "//*[local-name()='Class'][@*[local-name()='about']='http://purl.obolibrary.org/obo/DOID_8545']/*[local-name()='label']/text()" doid.owl

malignant hyperthermia


**Expected Output:**
malignant hyperthermia

We added the `@*[local-name()='label']` to select the element within the class that describes the label.
The output should be the label we were expecting: malignant hyperthermia.

We can do the same to get the label of the compound entry with the identifier `27732`:

In [23]:
%%bash
xmllint --xpath "//*[local-name()='Class'][@*[local-name()='about']='http://purl.obolibrary.org/obo/CHEBI_27732']/*[local-name()='label']/text()" chebi_lite.owl

caffeine


**Expected Output:**
caffeine

Again, the output should be the label we were expecting:
caffeine. We can now write a script that receives multiple URIs given as standard input and the OWL file where to find the labels. We can create a script named getlabels.sh

In [24]:
%%bash
cat << 'EOF' > getlabels.sh
OWLFILE=$1
xargs -I {} xmllint --xpath "//*[local-name()='Class'][@*[local-name()='about']='{}']/*[local-name()='label']/text()" $OWLFILE
EOF

The `xargs` command is used to process each line of the standard input. Now to execute the script we only need to provide the URIs as standard input:

In [25]:
%%bash
chmod u+x getlabels.sh

**Expected Output:**
No output.

In [26]:
%%bash
echo 'http://purl.obolibrary.org/obo/DOID_8545' | ./getlabels.sh doid.owl

malignant hyperthermia


**Expected Output:**
malignant hyperthermia

In [27]:
%%bash
echo 'http://purl.obolibrary.org/obo/CHEBI_27732' | ./getlabels.sh chebi_lite.owl

caffeine


The output should be the labels of those classes:

- malignant hyperthermia
- caffeine

We can also execute the script with multiple URIs:

In [28]:
%%bash
echo -e 'http://purl.obolibrary.org/obo/DOID_8545\nhttp://purl.obolibrary.org/obo/DOID_66' | ./getlabels.sh doid.owl

malignant hyperthermia
muscle tissue disease


**Expected Output:**
Labels for the provided DOID URIs.

In [29]:
%%bash
echo -e 'http://purl.obolibrary.org/obo/CHEBI_27732\nhttp://purl.obolibrary.org/obo/CHEBI_26385\nhttp://purl.obolibrary.org/obo/CHEBI_27134' | ./getlabels.sh chebi_lite.owl

caffeine
purine alkaloid
trimethylxanthine


**Expected Output:**
Labels for the provided CHEBI URIs.

The output will be a label for each URI.

To test both scripts, we can feed the output of one as the input of the other, for example:

In [30]:
%%bash
echo -e 'malignant hyperthermia\nmuscle tissue disease' | ./geturi.sh doid.owl | ./getlabels.sh doid.owl

malignant hyperthermia
muscle tissue disease


**Expected Output:** The same labels given as input.

In [31]:
%%bash
echo -e 'caffeine\npurine alkaloid\ntrimethylxanthine' | ./geturi.sh chebi_lite.owl | ./getlabels.sh chebi_lite.owl

caffeine
purine alkaloid
trimethylxanthine


**Expected Output:** The same labels given as input.

The output will be the original input, i.e. the labels given as arguments to the `echo` command.

Now we can use the URIs as input:

In [32]:
%%bash
echo -e 'http://purl.obolibrary.org/obo/DOID_8545\nhttp://purl.obolibrary.org/obo/DOID_66' | ./getlabels.sh doid.owl | ./geturi.sh doid.owl

http://purl.obolibrary.org/obo/DOID_8545
http://purl.obolibrary.org/obo/DOID_66


**Expected Output:**
The original URIs.

In [33]:
%%bash
echo -e 'http://purl.obolibrary.org/obo/CHEBI_27732\nhttp://purl.obolibrary.org/obo/CHEBI_26385\nhttp://purl.obolibrary.org/obo/CHEBI_27134' | ./getlabels.sh chebi_lite.owl | ./geturi.sh chebi_lite.owl

http://purl.obolibrary.org/obo/CHEBI_27732
http://purl.obolibrary.org/obo/CHEBI_26385
http://purl.obolibrary.org/obo/CHEBI_27134


**Expected Output:**
The original URIs.

Again the output will be the original input, i.e. the URIs given as arguments to the `echo` command.

## Synonyms

For example, to find all the synonyms of a disease, we can use the same XPath as used before but replacing the keyword label by `hasExactSynonym`:

In [34]:
%%bash
xmllint --xpath "//*[local-name()='Class'][@*[local-name()='about']='http://purl.obolibrary.org/obo/DOID_8545']/*[local-name()='hasExactSynonym']" doid.owl

<oboInOwl:hasExactSynonym xml:lang="en">anesthesia related hyperthermia</oboInOwl:hasExactSynonym>
<oboInOwl:hasExactSynonym xml:lang="en">malignant hyperpyrexia due to anesthesia</oboInOwl:hasExactSynonym>


**Expected Output:**
XML elements for synonyms (e.g., "malignant hyperpyrexia due to anesthesia").

The output will be the two synonyms of malignant hyperthermia

We can also get both the primary label and the synonyms. We only need to add an alternative match to the keyword label:

In [35]:
%%bash
xmllint --xpath "//*[local-name()='Class'][@*[local-name()='about']='http://purl.obolibrary.org/obo/DOID_8545']/*[local-name()='hasExactSynonym' or local-name()='label']" doid.owl

<oboInOwl:hasExactSynonym xml:lang="en">anesthesia related hyperthermia</oboInOwl:hasExactSynonym>
<oboInOwl:hasExactSynonym xml:lang="en">malignant hyperpyrexia due to anesthesia</oboInOwl:hasExactSynonym>
<rdfs:label rdf:datatype="http://www.w3.org/2001/XMLSchema#string">malignant hyperthermia</rdfs:label>


**Expected Output:**
XML elements for both label and synonyms.

The output will include now the two synonyms plus the official label

Thus, we can now update the script `getlabels.sh`:

In [36]:
%%bash
cat << 'EOF' > getlabels.sh
OWLFILE=$1
xargs -I {} xmllint --xpath "//*[local-name()='Class'][@*[local-name()='about']='{}']/*[local-name()='hasExactSynonym' or local-name()='hasRelatedSynonym' or local-name()='label']/text()" $OWLFILE
EOF
chmod u+x getlabels.sh

We can test the script exactly in the same way as before:

In [37]:
%%bash
echo -e 'http://purl.obolibrary.org/obo/DOID_8545' | ./getlabels.sh doid.owl

anesthesia related hyperthermia
malignant hyperpyrexia due to anesthesia
malignant hyperthermia


**Expected Output:**
Multiple labels (main label + synonyms) for malignant hyperthermia.

But now the output will display multiple labels for this class.

#### URI of synonyms

Since the script now returns alternative labels, we may encounter some problems if we send the output to the `geturi.sh` script:

In [38]:
%%bash
echo 'http://purl.obolibrary.org/obo/DOID_8545' | ./getlabels.sh doid.owl | ./geturi.sh doid.owl

http://purl.obolibrary.org/obo/DOID_8545


XPath set is empty
XPath set is empty


**Expected Output:**
The correct URI for the main label, followed by some XPath warnings.

The previous command will display XPath warnings for the two synonyms. If we do not want to know about these mismatches, we can always redirect them to the null device:

In [39]:
%%bash
echo 'http://purl.obolibrary.org/obo/DOID_8545' | ./getlabels.sh doid.owl | ./geturi.sh doid.owl 2>/dev/null

http://purl.obolibrary.org/obo/DOID_8545


**Expected Output:**
The URI(s) that could be resolved.

However, we can update the script `geturi.sh`:

In [40]:
%%bash
cat << 'EOF' > geturi.sh
OWLFILE=$1
xargs -I {} xmllint --xpath "//*[(local-name()='hasExactSynonym' or local-name()='hasRelatedSynonym' or local-name()='label') and text()='{}']/../@*[local-name()='about']" $OWLFILE | cut -d'"' -f2
EOF
chmod u+x geturi.sh

Now we can execute the same command:

In [41]:
%%bash
echo 'http://purl.obolibrary.org/obo/DOID_8545' | ./getlabels.sh doid.owl | ./geturi.sh doid.owl

http://purl.obolibrary.org/obo/DOID_8545
http://purl.obolibrary.org/obo/DOID_8545
http://purl.obolibrary.org/obo/DOID_8545


**Expected Output:**
The URI repeated for each synonym and label found.

Every label should now be matched exactly with the same class.

If we want to avoid repetitions, we can add the sort command with the `-u` option to the end of each command, as we did previously.

In [42]:
%%bash
echo 'http://purl.obolibrary.org/obo/DOID_8545' | ./getlabels.sh doid.owl | ./geturi.sh doid.owl | sort -u

http://purl.obolibrary.org/obo/DOID_8545


**Expected Output:** Now only one URI.

## Parent Classes

Parent classes represent generalizations that may also be relevant to recognize in text. To extract all the parent classes of malignant hyperthermia, we can use the following XPath query:

In [43]:
%%bash
xmllint --xpath "//*[local-name()='Class'][@*[local-name()='about']='http://purl.obolibrary.org/obo/DOID_8545']/*[local-name()='subClassOf']/@*[local-name()='resource']" doid.owl

 rdf:resource="http://purl.obolibrary.org/obo/DOID_0050736"
 rdf:resource="http://purl.obolibrary.org/obo/DOID_66"


**Expected Output:**
Resource attributes pointing to parent classes.

The first part of the XPath is the same as the above to get the class element, then `[local-name()='subClassOf']` is used to get the subclass element, and finally `@*[local-name()='resource']` is used to get the attribute containing its URI.

The output should be the URIs representing the parents of class `8545`.

We can also execute the same command for caffeine:

In [44]:
%%bash
xmllint --xpath "//*[local-name()='Class'][@*[local-name()='about']='http://purl.obolibrary.org/obo/CHEBI_27732']/*[local-name()='subClassOf']/@*[local-name()='resource']" chebi_lite.owl

 rdf:resource="http://purl.obolibrary.org/obo/CHEBI_26385"
 rdf:resource="http://purl.obolibrary.org/obo/CHEBI_27134"


**Expected Output:**
Resource attributes pointing to parent classes of caffeine.

Note that we no longer can use the string function, because ontologies are organized as DAGs using multiple inheritance, i.e. each class can have multiple parents, and the string function only returns the first match. To get only the URIs, we can apply the previous technique of using the `cut` command:

In [45]:
%%bash
xmllint --xpath "//*[local-name()='Class'][@*[local-name()='about']='http://purl.obolibrary.org/obo/DOID_8545']/*[local-name()='subClassOf']/@*[local-name()='resource']" doid.owl | cut -d'"' -f2

http://purl.obolibrary.org/obo/DOID_0050736
http://purl.obolibrary.org/obo/DOID_66


**Expected Output:**
Only the URIs of the parents.

We can now create a script that receives multiple URIs as standard input and the OWL file as an argument. The script is named `getparents.sh`:

In [46]:
%%bash
cat << 'EOF' > getparents.sh
OWLFILE=$1
xargs -I {} xmllint --xpath "//*[local-name()='Class'][@*[local-name()='about']='{}']/*[local-name()='subClassOf']/@*[local-name()='resource']" $OWLFILE | cut -d'"' -f2
EOF

and add the permission:

In [47]:
%%bash
chmod u+x getparents.sh

To get the parents of malignant hyperthermia, we will only need to give the URI as input and the OWL file as argument:

In [48]:
%%bash
echo 'http://purl.obolibrary.org/obo/DOID_8545' | ./getparents.sh doid.owl

http://purl.obolibrary.org/obo/DOID_0050736
http://purl.obolibrary.org/obo/DOID_66


**Expected Output:**
URIs of the two parents of malignant hyperthermia.

The output will include the URIs of the two parents.

#### Labels of parents

But if we need the labels we can redirect the output to the `getlabels.sh` script:

In [49]:
%%bash
echo 'http://purl.obolibrary.org/obo/DOID_8545' | ./getparents.sh doid.owl | ./getlabels.sh doid.owl

autosomal dominant disease
muscle tissue disease


**Expected Output:**
Labels of the parents (e.g. "muscle tissue disease").

The output will now be the label of the parents of malignant hyperthermia.

Again, the same can be done with caffeine:

In [50]:
%%bash
echo 'http://purl.obolibrary.org/obo/CHEBI_27732' | ./getparents.sh chebi_lite.owl | ./getlabels.sh chebi_lite.owl

purine alkaloid
trimethylxanthine


**Expected Output:**
Labels of the parents of caffeine.

And now the output contains the labels of the parents of caffeine.

#### Related classes

If we are interested in using all the related classes besides the ones that represent a generalization (`subClassOf` ), we have to change our XPath to:

In [51]:
%%bash
xmllint --xpath "//*[local-name()='Class'][@*[local-name()='about']='http://purl.obolibrary.org/obo/CHEBI_27732']/*[local-name()='subClassOf']//*[local-name()='someValuesFrom']/@*[local-name()='resource']" chebi_lite.owl | cut -d'"' -f2

http://purl.obolibrary.org/obo/CHEBI_25435
http://purl.obolibrary.org/obo/CHEBI_35337
http://purl.obolibrary.org/obo/CHEBI_35471
http://purl.obolibrary.org/obo/CHEBI_35498
http://purl.obolibrary.org/obo/CHEBI_35703
http://purl.obolibrary.org/obo/CHEBI_50218
http://purl.obolibrary.org/obo/CHEBI_50925
http://purl.obolibrary.org/obo/CHEBI_53121
http://purl.obolibrary.org/obo/CHEBI_60809
http://purl.obolibrary.org/obo/CHEBI_64047
http://purl.obolibrary.org/obo/CHEBI_67114
http://purl.obolibrary.org/obo/CHEBI_71232
http://purl.obolibrary.org/obo/CHEBI_75771
http://purl.obolibrary.org/obo/CHEBI_76924
http://purl.obolibrary.org/obo/CHEBI_76946
http://purl.obolibrary.org/obo/CHEBI_78298
http://purl.obolibrary.org/obo/CHEBI_85234


**Expected Output:**
URIs of related classes (e.g. mutagen).

Keep in mind that these related classes are in the attribute resource of `someValuesFrom` element inside a subClassOf element. The URIs of the related classes of caffeine are now displayed.

#### Labels of related classes

To get the labels of these related classes, we only need to add the getlabels.sh script:

In [52]:
%%bash
xmllint --xpath "//*[local-name()='Class'][@*[local-name()='about']='http://purl.obolibrary.org/obo/CHEBI_27732']/*[local-name()='subClassOf']//*[local-name()='someValuesFrom']/@*[local-name()='resource']" chebi_lite.owl | cut -d'"' -f2 | ./getlabels.sh chebi_lite.owl

mutagen
central nervous system stimulant
psychotropic drug
diuretic
xenobiotic
EC 3.1.4.* (phosphoric diester hydrolase) inhibitor
EC 2.7.11.1 (non-specific serine/threonine protein kinase) inhibitor
adenosine A2A receptor antagonist
adjuvant
food additive
ryanodine receptor agonist
adenosine receptor antagonist
mouse metabolite
plant metabolite
fungal metabolite
environmental contaminant
human blood serum metabolite


**Expected Output:**
Labels of related classes (e.g. "mutagen").

The output is now terms that we could use to expand our text processing.

## Ancestors

Finding all the ancestors of a class includes many chain invocations of the `getparents.sh` until we get no matches. We also should avoid relations that are cyclic, otherwise we will enter in a infinite loop. Thus, for identifying the ancestors of a class, we will only consider parent relations, i.e. subsumption relations.

#### Grandparents

In the previous step we were able to extract the direct parents of a class, but the parents of these parents also represent generalizations of the original class. For example, to get the parents of the parents (grandparents) of malignant hyperthermia we need to invoke getparents.sh twice:

In [53]:
%%bash
echo 'malignant hyperthermia' | ./geturi.sh doid.owl | ./getparents.sh doid.owl | ./getparents.sh doid.owl

http://purl.obolibrary.org/obo/DOID_0050739
http://purl.obolibrary.org/obo/DOID_0080000


**Expected Output:**
URIs of the grandparents.

And we will find the URIs of the grandparents of malignant hyperthermia.

Or to get their labels we can add the `getlabels.sh` script:

In [54]:
%%bash
echo 'malignant hyperthermia' | ./geturi.sh doid.owl | ./getparents.sh doid.owl | ./getparents.sh doid.owl | ./getlabels.sh doid.owl

autosomal genetic disease
muscular disease


**Expected Output:**
Labels of the grandparents.

And we find the labels of the grandparents of malignant hyperthermia.

#### Root class

However, there are classes that do not have any parent, which are called root classes. _disease_ and _chemical entity_ are root classes of DO and ChEBI ontologies, respectively. As we can see these are highly generic terms.
To check if it is the root class, we can ask for their parents:

In [55]:
%%bash
echo 'disease' | ./geturi.sh doid.owl | ./getparents.sh doid.owl

XPath set is empty


**Expected Output:**
XPath set is empty (or similar warning).

In [56]:
%%bash
echo 'chemical entity' | ./geturi.sh chebi_lite.owl | ./getparents.sh chebi_lite.owl

XPath set is empty


**Expected Output:**
XPath set is empty (or similar warning).

In both cases, we will get the warning that no matches were found, confirming that they are the root class.

```text
XPath set is empty
```

#### Recursion

We can now build a script that receives a list of URIs as standard input, and invokes `getparents.sh` recursively until it reaches the root class. The script named `getancestors.sh`

In [57]:
%%bash
cat << 'EOF' > getancestors.sh
OWLFILE=$1
CLASSES=$(cat -)
[[ -z "$CLASSES" ]] && exit
PARENTS=$(echo "$CLASSES" | ./getparents.sh $OWLFILE | sort -u)
echo "$PARENTS"
echo "$PARENTS" | ./getancestors.sh $OWLFILE
EOF

The second line of the script saves the standard input in a variable named `CLASSES`, because we need to use it twice: i) to check if the input as any classes or is empty (line 3) and ii) to get the parents of the classes given as input (line 4). If the input is empty then the script ends, this is the base case of the [recursion](https://en.wikipedia.org/wiki/Recursion). This is required so the recursion stops at a given point. Otherwise, the script would run indefinitely until the user stops it manually. The fourth line of the script stores the output in a variable named `PARENTS`, because we need also to use it twice: i) to output these direct parents (line 5), and ii) to get the ancestors of these parents (line 6). Note that we are invoking the `getancestors.sh` script inside the `getancestors.sh`, which defines the recursion step. Since the subsumption relation is acyclic, we expect that at some time we will reach classes without parents (root classes) and then the script will end.
Importantly, the echo of the variables `CLASSES` and `PARENTS` need to be inside double quotes, so the newline characters are preserved.

#### Iteration

Recursion is most of the times computational expensive, but usually it is possible to replace recursion with iteration to develop a more efficient algorithm.
Explaining iteration and how to refactor a recursive script is out of scope of this tutorial, nevertheless the following script represents an equivalent way to get all the ancestors without using recursion:

```bash
OWLFILE=$1
CLASSES=$(cat -)
ANCESTORS=""
while [[ ! -z "$CLASSES" ]]
do
  PARENTS=$(echo "$CLASSES" | ./getparents.sh $OWLFILE | sort -u)
  ANCESTORS="$ANCESTORS\n$PARENTS"
  CLASSES=$PARENTS
done
echo -e "$ANCESTORS"
```

The script uses the while command that basically implements iteration by repeating a set of commands (lines 6-8) while a given condition is satisfied (line 4).

To test the recursive script, we can provide as standard input the label malignant hyperthermia:

In [58]:
%%bash
chmod u+x getancestors.sh
echo 'http://purl.obolibrary.org/obo/DOID_8545' | ./getancestors.sh doid.owl

http://purl.obolibrary.org/obo/DOID_0050736
http://purl.obolibrary.org/obo/DOID_66
http://purl.obolibrary.org/obo/DOID_0050739
http://purl.obolibrary.org/obo/DOID_0080000
http://purl.obolibrary.org/obo/DOID_0050177
http://purl.obolibrary.org/obo/DOID_17
http://purl.obolibrary.org/obo/DOID_630
http://purl.obolibrary.org/obo/DOID_7
http://purl.obolibrary.org/obo/DOID_4



XPath set is empty


**Expected Output:**
List of URIs for all ancestors.

The output will be the URI of all its ancestors

Note that we will still receive the XPath warning when the script reaches the root class and no parents are found:

```text
XPath set is empty
```

To remove this warning and just get the labels of the ancestors of malignant hyperthermia, we can redirect the warnings to the null device:

In [59]:
%%bash
echo 'malignant hyperthermia' | ./geturi.sh doid.owl | ./getancestors.sh doid.owl 2>/dev/null | ./getlabels.sh doid.owl

autosomal dominant disease
muscle tissue disease
autosomal genetic disease
muscular disease
monogenic disease
musculoskeletal system disease
genetic disease
disease of anatomical entity
disease


**Expected Output:**
Labels of all ancestors.

The output will now include the name of all ancestors of malignant hyperthermia.
Note that the first two ancestors are the direct parents of malignant hyperthermia, and the last one is the root class. This happens because the recursive script prints the parents before invoking itself to find the ancestors of the direct parents.
We can do the same with caffeine, but be advised that given the higher number of ancestors in ChEBI we may now have to wait a little longer for the script to end.

In [60]:
%%bash
echo 'caffeine' | ./geturi.sh chebi_lite.owl | ./getancestors.sh chebi_lite.owl | ./getlabels.sh chebi_lite.owl | sort -u

alkaloid
aromatic compound
bicyclic compound
carbon group molecular entity
chemical entity
cyclic compound
heteroarene
heterobicyclic compound
heterocyclic compound
heteroorganic entity
heteropolycyclic compound
imidazopyrimidine
main group molecular entity
methylxanthine
molecular entity
molecule
nitrogen molecular entity
organic aromatic compound
organic cyclic compound
organic heterobicyclic compound
organic heterocyclic compound
organic heteropolycyclic compound
organic molecular entity
organic molecule
organonitrogen compound
organonitrogen heterocyclic compound
p-block molecular entity
pnictogen molecular entity
polyatomic entity
polycyclic compound
purine alkaloid
purines
trimethylxanthine


XPath set is empty
XPath set is empty
XPath set is empty
XPath set is empty
XPath set is empty
XPath set is empty
XPath set is empty
XPath set is empty


**Expected Output:**
Labels of all ancestors of caffeine, sorted unique.

The results include repeated classes that were found by using different branches, so that is why we need to add the sort command with the `-u` option to eliminate the duplicates. The script will print the ancestors being found by the script.

## My Lexicon

Now that we know how to extract all the labels and related classes from an ontology, we can construct our own lexicon with the list of terms that we want to recognize in text.
Let us start by creating the file `do_8545_lexicon.txt` representing our lexicon for malignant hyperthermia with all its labels:

In [61]:
%%bash
echo 'malignant hyperthermia' | ./geturi.sh doid.owl | ./getlabels.sh doid.owl > do_8545_lexicon.txt

**Expected Output:**
No output (file created).

#### Ancestors labels

Now we can add to the lexicon all the labels of the ancestors of malignant hyperthermia by adding the redirection operator:

In [62]:
%%bash
echo 'malignant hyperthermia' | ./geturi.sh doid.owl | ./getancestors.sh doid.owl | ./getlabels.sh doid.owl >> do_8545_lexicon.txt

XPath set is empty


**Expected Output:**
No output (file appended).

Note that now we use `>>` and not `>`, this will append more lines to the file instead of creating a new file from scratch.
Now we can check the contents of the file `do_8545_lexicon.txt` to see the terms we got:

In [63]:
%%bash
cat do_8545_lexicon.txt | sort -u

anesthesia related hyperthermia
autosomal dominant disease
autosomal genetic disease
disease
disease of anatomical entity
genetic disease
malignant hyperpyrexia due to anesthesia
malignant hyperthermia
monogenic disease
muscle tissue disease
muscular disease
musculoskeletal system disease


**Expected Output:**
List of labels in the lexicon.

Note that we use the sort command with the `-u` option to eliminate any duplicates that may exist.

We can also apply the same commands for caffeine to produce its lexicon in the file `chebi_27732_lexicon.txt` by adding the redirection operator:

In [64]:
%%bash
echo 'caffeine' | ./geturi.sh chebi_lite.owl | ./getlabels.sh chebi_lite.owl > chebi_27732_lexicon.txt
echo 'caffeine' | ./geturi.sh chebi_lite.owl | ./getancestors.sh chebi_lite.owl | ./getlabels.sh chebi_lite.owl >> chebi_27732_lexicon.txt

XPath set is empty
XPath set is empty
XPath set is empty
XPath set is empty
XPath set is empty
XPath set is empty
XPath set is empty
XPath set is empty


**Expected Output:**
No output (files created/appended).

> Please note that it may take some time to retrieve all labels, even with these reduced OWL files. In the meantime, you can look at the following steps while it is running.

Now let us check the contents of this new lexicon:

In [65]:
%%bash
cat chebi_27732_lexicon.txt | sort -u

alkaloid
aromatic compound
bicyclic compound
caffeine
carbon group molecular entity
chemical entity
cyclic compound
heteroarene
heterobicyclic compound
heterocyclic compound
heteroorganic entity
heteropolycyclic compound
imidazopyrimidine
main group molecular entity
methylxanthine
molecular entity
molecule
nitrogen molecular entity
organic aromatic compound
organic cyclic compound
organic heterobicyclic compound
organic heterocyclic compound
organic heteropolycyclic compound
organic molecular entity
organic molecule
organonitrogen compound
organonitrogen heterocyclic compound
p-block molecular entity
pnictogen molecular entity
polyatomic entity
polycyclic compound
purine alkaloid
purines
trimethylxanthine


**Expected Output:**
List of labels for caffeine lexicon.

Now we should be able to see that this lexicon is much larger.

#### Merging labels

If we are interested in finding everything related to caffeine or malignant hyperthermia, we may be interested in merging the two lexicons in a file named `lexicon.txt`:

In [66]:
%%bash
cat do_8545_lexicon.txt chebi_27732_lexicon.txt | sort -u > lexicon.txt

**Expected Output:**
No output (file created).

Using this new lexicon, we can recognize mentions in any text file.
To get started, we first need to retrieve the data file generated in the previous tutorial. The following command downloads the `chebi_27732_sentences.txt` file directly from the GitHub repository:

In [67]:
%%bash
curl -s -O 'https://raw.githubusercontent.com/lasigeBioTM/data-text-processing-notebooks/refs/heads/main/data/chebi_27732_sentences.txt'

Using this new lexicon, we can recognize any mention in our previous file named `chebi_27732_sentences.txt`:

In [68]:
%%bash
grep -w -i -F -f lexicon.txt chebi_27732_sentences.txt

Mutation screening of the RYR1 gene and identification of two novel mutations in Italian malignant hyperthermia families.
Point mutations in the ryanodine receptor (RYR1) gene are associated with malignant hyperthermia, an autosomal dominant disorder triggered in susceptible people (MHS) by volatile anaesthetics and depolarising skeletal muscle relaxants.
A mutation in the transmembrane/luminal domain of the ryanodine receptor is associated with abnormal Ca(2+) release channel function and severe central core disease.
Central core disease is a rare, nonprogressive myopathy that is characterized by hypotonia and proximal muscle weakness.
 The response of the mutant RyR1 Ca2+ channel to the agonists halothane and caffeine in a Ca2+ photometry assay was completely abolished.
 Coexpression of normal and mutant RYR1 cDNAs in a 1:1 ratio, however, produced RyR1 channels with normal halothane and caffeine sensitivities, but maximal levels of Ca2+ release were reduced by 67%.
 Comparison with 

**Expected Output:**
Sentences matching terms in the lexicon.

We added the `-F` option because our lexicon is a list of fixed strings, i.e. does not include regular expressions. The equivalent long form to the `-F` option is `--fixed-strings`.
We now get more sentences, including some that do not include a direct mention to caffeine or malignant hyperthermia. For example, the following sentence was selected because it mentions molecule, which is an ancestor of caffeine.

```text
The remainder of the molecule is hydrophilic
and presumably constitutes the cytoplasmic
domain of the protein.
```

Another example is the following sentence, which was selected because it mentions disease, which is an ancestor of malignant hyperthermia:

```text
Our data suggest that divergent activity profiles
may cause varied disease phenotypes by specific mutations.
```

We can also use our script `getentities.sh` (previous tutorial) giving this lexicon as argument. However, since we are not using any regular expressions it would be better to replace the `-E` option by `-F` to the `grep` command in the script, so the lexicon is interpreted as list of fixed strings to be matched. Only then we can execute the script safely.

#### Ancestors matched

Besides these two previous examples, we can check if there other ancestors being matched by using the grep command with the `-o` option:

In [69]:
%%bash
grep -o -w -F -f lexicon.txt chebi_27732_sentences.txt | sort -u

caffeine
disease
malignant hyperthermia
molecule


**Expected Output:**
List of matched terms.

We can see that besides the terms caffeine and malignant hyperthermia, only one ancestor of each one of them was matched, molecule and disease, respectively.

This can be explained because our text is somehow limited and because we are using the official labels and we may be missing acronyms, and simple variations such as the plural of a term. To cope with this issue, we may use a [stemmer](https://en.wikipedia.org/wiki/Stemming), or use all the ancestors besides subsumption. However, if our lexicon is small is better to do it manually and maybe add some regular expressions to deal with some of the variations.

## Generic Lexicon

Instead of using a customized and limited lexicon, we may be interested in recognizing any of the diseases represented in the ontology. By recognizing all the diseases in our caffeine related text, we will be able to find all the diseases that may be related to caffeine

#### All labels

To extract all the labels from the disease ontology we can use the same XPath query used before, but now without restricting it to any URI:

In [70]:
%%bash
xmllint --xpath "//*[local-name()='Class']/*[local-name()='hasExactSynonym' or local-name()='hasRelatedSynonym' or local-name()='label']/text()" doid.owl

monogenic disease
Post measles encephalitis (disorder)
Post-measles encephalitis
postmeasles encephalitis
obsolete Measles virus encephalitis
BENIGN CHRONIC PEMPHIGUS
Pemphigus, Benign Familial
Hailey-Hailey disease
ARVC
ARVC cardiomyopathy
ARVD
arrhythmogenic right ventricular dysplasia
arrhythmogenic right ventricular dysplasia/cardiomyopathy
arrhythmogenic right ventricular cardiomyopathy
ANDERSEN CARDIODYSRHYTHMIC PERIODIC PARALYSIS
Andersen syndrome
LQT7
Long QT syndrome 7
Potassium-Sensitive Cardiodysrhythmic Type
Andersen-Tawil syndrome
Cardiomyopathies
cardiomyopathy
autosomal dominant disease
autosomal genetic disease
Zollinger-Ellison syndrome
syndromic intellectual disability
A-fib
atrial fibrillation
scoliosis
cardiopulmonary arrest
circulatory arrest
cardiac arrest
Ziziphus mauritiana fruit allergy
Indian plum allergy
catecholaminergic polymorphic ventricular tachycardia
CLPED1
Margarita type of ectodermal dysplasia
Zlotogora-Zilberman-Tenenbaum syndrome
cleft lip/palate-s

**Expected Output:**
Huge list of labels.

We can create a script named `getalllabels.sh`:

In [71]:
%%bash
cat << 'EOF' > getalllabels.sh
OWLFILE=$1
xmllint --xpath "//*[local-name()='Class']/*[local-name()='hasExactSynonym' or local-name()='hasRelatedSynonym' or local-name()='label']/text()" $OWLFILE | sort -u
EOF

**Expected Output:**
No output (file created).

That receives as argument the OWL file where to find all labels.
Note that this script is similar to the `getlabels.sh` script without the `xargs`, since it does not receive a list of URIs as standard input.
Now we can execute the script to extract all labels from the OWL file:

In [72]:
%%bash
chmod u+x getalllabels.sh
./getalllabels.sh doid.owl

46,XY disorder of sex development due to LHB deficiency
46,XY disorder of sex development due to luteinizing hormone subunit beta deficiency
46,XY DSD due to LHB deficiency
46,XY DSD due to luteinizing hormone subunit beta deficiency
47, XXY
AD2
A-fib
alpha thalassemia-intellectual disability syndrome, deletion type
alpha-thalassemia-intellectual disability syndrome linked to chromosome 16
alpha thalassemia-intellectual disability syndrome type 1
alpha-thalassemia/mental retardation syndrome, deletion-type
alpha-thalassemia/mental retardation syndrome nondeletion type
alpha-thalassemia/mental retardation syndrome, type 1
alpha thalassemia-retardation syndrome
alpha thalassemia-X-linked intellectual disability syndrome
Alzheimer disease-2
Alzheimer disease 2, late onset
Alzheimer disease associated with APOE4
Alzheimer's disease 2
ANDERSEN CARDIODYSRHYTHMIC PERIODIC PARALYSIS
Andersen syndrome
Andersen-Tawil syndrome
anesthesia related hyperthermia
animal phobia
arrhythmogenic right ven

**Expected Output:**
List of all labels in doid.owl.

The output will contain the full list of diseases in this reduced version of the ontology.

To create the generic lexicon, we can redirect the output to the file diseases.txt:

In [73]:
%%bash
./getalllabels.sh doid.owl > diseases.txt

**Expected Output:**
No output (file created).

We can check how many labels we got by using the wc command:

In [74]:
%%bash
wc -l diseases.txt

224 diseases.txt


**Expected Output:**
Line count of diseases.txt.

From the original OWL file the lexicon contains thousands of labels.

We can now recognize the lexicon entries in the sentences of the file `chebi_27732_sentences.txt` by using the grep command:

In [75]:
%%bash
grep -n -w -F -f diseases.txt chebi_27732_sentences.txt

1:Mutation screening of the RYR1 gene and identification of two novel mutations in Italian malignant hyperthermia families.
2:Point mutations in the ryanodine receptor (RYR1) gene are associated with malignant hyperthermia, an autosomal dominant disorder triggered in susceptible people (MHS) by volatile anaesthetics and depolarising skeletal muscle relaxants.
9:A mutation in the transmembrane/luminal domain of the ryanodine receptor is associated with abnormal Ca(2+) release channel function and severe central core disease.
10:Central core disease is a rare, nonprogressive myopathy that is characterized by hypotonia and proximal muscle weakness.
19: Comparison with two other coexpressed mutant/normal channels suggests that the I4898T mutation produces one of the most abnormal RyR1 channels yet investigated, and this level of abnormality is reflected in the severe and penetrant phenotype of affected central core disease individuals.
31: Four of them are also associated with central core

**Expected Output:**
Matched sentences with line numbers.

The output will show the large list of sentences mentioning diseases.

Problematic entries

Despite using the `-F` option, the lexicon contains some problematic entries.
Some entries have expressions enclosed by parentheses or brackets, that represent alternatives or a category:

```text
Post measles encephalitis (disorder)
Glaucomatous atrophy [cupping] of optic disc
```

Other entries have separation characters, such as commas or colons, to represent a specialization. For example:

```text
Tapeworm infection: intestinal taenia solum
Tapeworm infection: pork
Pemphigus, Benign Familial
ATR, nondeletion type
```

A problem is that not all have the same meaning. A comma may also be part of the term. For example:

```text
46,XY DSD due to LHB deficiency
```

Other case includes using `&amp;` to represent an ampersand. For example:

```text
Gonococcal synovitis &amp;/or tenosynovitis
```

However, most of the times the alternatives are already included in the lexicon in different lines. For example:

```text
Gonococcal synovitis and tenosynovitis
Gonococcal synovitis or tenosynovitis
```

As we can see by these examples, it is not trivial to devise rules that fully solve these issues. Very likely there will be exceptions to any rule we devise and that we are not aware of.

#### Special characters frequency

To check the impact of each of these issues, we can count the number of times they appear in the lexicon:

In [76]:
%%bash
grep -c -F '(' diseases.txt
grep -c -F ',' diseases.txt
grep -c -F '[' diseases.txt
grep -c -F ':' diseases.txt
grep -c -F '&amp;' diseases.txt

2
17
1
2
1


**Expected Output:**
Counts of special characters.

In the original OWL file we would see that parentheses and commas are the most frequent, with more than one thousand entries.

#### Completeness

Now let us check if the ATR acronym representing the _alpha thalassemia-X- linked intellectual disability syndrome_ is in the lexicon:

In [77]:
%%bash
grep -E '^ATR' diseases.txt

ATR-16 syndrome
ATR, nondeletion type
ATR syndrome, deletion type
ATR syndrome linked to chromosome 16
ATR-X syndrome


**Expected Output:**
Entries starting with ATR (likely none just 'ATR').

All the entries include more terms than only the acronym.

Thus, a single ATR mention will not be recognized.
This is problematic if we need to match sentences mentioning that acronym, such as:

In [78]:
%%bash
echo 'The ATR syndrome is an alpha thalassemia that has material basis in mutation in the ATRX gene on Xq21' | grep -w 'ATR'

The ATR syndrome is an alpha thalassemia that has material basis in mutation in the ATRX gene on Xq21


**Expected Output:** The sentence.

We will now try to mitigate these issues as simply as we can. We will not try to solve them completely, but at least address the most obvious cases.

#### Removing special characters

The first fix we will do, is to remove all the parentheses and brackets by using the tr command, since they will not be found in the text:

In [79]:
%%bash
tr -d '[](){}' < diseases.txt

46,XY disorder of sex development due to LHB deficiency
46,XY disorder of sex development due to luteinizing hormone subunit beta deficiency
46,XY DSD due to LHB deficiency
46,XY DSD due to luteinizing hormone subunit beta deficiency
47, XXY
AD2
A-fib
alpha thalassemia-intellectual disability syndrome, deletion type
alpha-thalassemia-intellectual disability syndrome linked to chromosome 16
alpha thalassemia-intellectual disability syndrome type 1
alpha-thalassemia/mental retardation syndrome, deletion-type
alpha-thalassemia/mental retardation syndrome nondeletion type
alpha-thalassemia/mental retardation syndrome, type 1
alpha thalassemia-retardation syndrome
alpha thalassemia-X-linked intellectual disability syndrome
Alzheimer disease-2
Alzheimer disease 2, late onset
Alzheimer disease associated with APOE4
Alzheimer's disease 2
ANDERSEN CARDIODYSRHYTHMIC PERIODIC PARALYSIS
Andersen syndrome
Andersen-Tawil syndrome
anesthesia related hyperthermia
animal phobia
arrhythmogenic right ven

**Expected Output:**
Lexicon content with brackets removed.

Of course, we may lose the shorter labels, such as _Post measles encephalitis_, but at least now, the disease _Post measles encephalitis disorder_ will be recognized:

In [80]:
%%bash
tr -d '[](){}' < diseases.txt | grep 'Post measles encephalitis disorder'

Post measles encephalitis disorder


**Expected Output:**
The cleaned label.

If we really need these alternatives, we would have to create multiple entries in the lexicon or transform the labels in regular expressions.

#### Removing extra terms

The second fix is to remove all the text after a separation character, by using the `sed` command:

In [81]:
%%bash
tr -d '[](){}' < diseases.txt | sed -E 's/[,:;] .*$//'

46,XY disorder of sex development due to LHB deficiency
46,XY disorder of sex development due to luteinizing hormone subunit beta deficiency
46,XY DSD due to LHB deficiency
46,XY DSD due to luteinizing hormone subunit beta deficiency
47
AD2
A-fib
alpha thalassemia-intellectual disability syndrome
alpha-thalassemia-intellectual disability syndrome linked to chromosome 16
alpha thalassemia-intellectual disability syndrome type 1
alpha-thalassemia/mental retardation syndrome
alpha-thalassemia/mental retardation syndrome nondeletion type
alpha-thalassemia/mental retardation syndrome
alpha thalassemia-retardation syndrome
alpha thalassemia-X-linked intellectual disability syndrome
Alzheimer disease-2
Alzheimer disease 2
Alzheimer disease associated with APOE4
Alzheimer's disease 2
ANDERSEN CARDIODYSRHYTHMIC PERIODIC PARALYSIS
Andersen syndrome
Andersen-Tawil syndrome
anesthesia related hyperthermia
animal phobia
arrhythmogenic right ventricular cardiomyopathy
arrhythmogenic right ventricula

**Expected Output:**
Lexicon cleaned of suffixes after separators.

Note that the regular expression enforces a space after the separation character to avoid separation characters that are not really separating two expressions, such as: _46,XY DSD due to LHB deficiency_
We can see that now we are able to recognize both ATR and _ATR syndrome_:

In [82]:
%%bash
tr -d '[](){}' < diseases.txt | sed -E 's/[,:;] .*$//' | grep -E '^ATR'

ATR-16 syndrome
ATR
ATR syndrome
ATR syndrome linked to chromosome 16
ATR-X syndrome


**Expected Output:**
ATR matches.

#### Removing extra spaces

The third fix is to remove any leading or trailing spaces of a label:

In [83]:
%%bash
tr -d '[](){}' < diseases.txt | sed -E 's/[,:;] .*$//; s/^ *//; s/ *$//'

46,XY disorder of sex development due to LHB deficiency
46,XY disorder of sex development due to luteinizing hormone subunit beta deficiency
46,XY DSD due to LHB deficiency
46,XY DSD due to luteinizing hormone subunit beta deficiency
47
AD2
A-fib
alpha thalassemia-intellectual disability syndrome
alpha-thalassemia-intellectual disability syndrome linked to chromosome 16
alpha thalassemia-intellectual disability syndrome type 1
alpha-thalassemia/mental retardation syndrome
alpha-thalassemia/mental retardation syndrome nondeletion type
alpha-thalassemia/mental retardation syndrome
alpha thalassemia-retardation syndrome
alpha thalassemia-X-linked intellectual disability syndrome
Alzheimer disease-2
Alzheimer disease 2
Alzheimer disease associated with APOE4
Alzheimer's disease 2
ANDERSEN CARDIODYSRHYTHMIC PERIODIC PARALYSIS
Andersen syndrome
Andersen-Tawil syndrome
anesthesia related hyperthermia
animal phobia
arrhythmogenic right ventricular cardiomyopathy
arrhythmogenic right ventricula

**Expected Output:**
Cleaned lexicon.

Note that we added two more replacement expressions to the `sed` command by separating them with a semicolon.

We can now update the script `getalllabels.sh` to include the previous `tr` and `sed` commands:

In [84]:
%%bash
cat << 'EOF' > getalllabels.sh
OWLFILE=$1

xmllint \
    --xpath \
        "//*[local-name()='Class']/*[local-name()='hasExactSynonym' or local-name()='hasRelatedSynonym' or local-name()='label']/text()" $OWLFILE | \
    tr -d '[](){}' | \
    sed -E 's/[,:;] .*$//; s/^ *//; s/ *$//' | \
    sort -u
EOF

**Expected Output:**
No output (file updated).

And we can now generate a fixed lexicon:

In [85]:
%%bash
./getalllabels.sh doid.owl > diseases.txt

**Expected Output:**
No output (file created).

We can check again the number of entries:

In [86]:
%%bash
wc -l diseases.txt

222 diseases.txt


**Expected Output:**
New line count.

We have less entries because our fixes made some entries equal to others already in the lexicon, and thus the `-u` option filtered them.
From the original OWL we would get a lexicon with more than 13 thousand labels.

#### Disease recognition

We can now try to recognize lexicon entries in the sentences of file `chebi_27732_sentences.txt`:

In [87]:
%%bash
grep -n -o -w -F -f diseases.txt chebi_27732_sentences.txt

1:malignant hyperthermia
2:malignant hyperthermia
9:central core disease
10:disease
10:myopathy
19:central core disease
31:central core disease
31:congenital myopathy
33:disease
36:disease
37:malignant hyperthermia
38:ATR
39:ATR
39:ATR
41:ATR
41:ATR
42:ATR
43:ATR
43:ATR
44:ATR
52:ATR
53:ATR
56:ATR
57:ATR
58:ATR
59:ATR
60:ATR
61:ATR
62:ATR
63:ATR
65:ATR
66:ATR
71:ATR
73:malignant hyperthermia
131:malignant hyperthermia
153:congenital myopathy
154:disease
154:nemaline rod myopathy
156:rod myopathy
158:congenital myopathy
158:nemaline myopathy
159:nemaline myopathy
159:disease
164:ATR
169:ATR
170:ATR
171:ATR
171:ATR
172:ATR
173:ATR
173:ATR
174:ATR
175:ATR
176:ATR
176:ATR
177:disease
178:disease
192:HL
196:ARVD2
196:cardiomyopathy
197:disease
198:ARVD2
200:malignant hyperthermia
200:central core disease
202:ARVD2
202:disease
203:arrhythmogenic right ventricular cardiomyopathy
203:ARVD2
228:cardiac arrest
230:disease
231:catecholaminergic polymorphic ventricular tachycardia
233:disease
237:

**Expected Output:**
Matched diseases with line numbers.

To obtain the list of labels that were recognized, we can use the grep command:

In [88]:
%%bash
grep -o -w -F -f diseases.txt chebi_27732_sentences.txt | sort -u

47
Andersen-Tawil syndrome
arrhythmogenic right ventricular cardiomyopathy
ARVD2
ataxia telangiectasia
ATR
atrial fibrillation
benign congenital myopathy
cancer
cardiac arrest
cardiomyopathy
catecholaminergic polymorphic ventricular tachycardia
central core disease
CFTD
chorea
congenital myopathy
disease
dystonia
epilepsy
FHL1
hepatitis C
HL
hypercholesterolaemia
hypokalemic periodic paralysis
Hypokalemic periodic paralysis
intellectual disability
long QT syndrome
LQT1
LQT2
LQT3
LQT5
LQT6
malignant hyperthermia
mental retardation
migraine
myopathy
myotonic dystrophy type 1
nemaline myopathy
nemaline rod myopathy
ophthalmoplegia
rod myopathy
scoliosis
syndrome


**Expected Output:**
Unique list of recognized diseases.

From the original OWL we would get a list of 47 unique labels representing diseases that may be related to caffeine:

The reason why `47` appears is because there is a label 47, XXY:

In [89]:
%%bash
echo '47, XXY' | ./geturi.sh doid.owl

http://purl.obolibrary.org/obo/DOID_1921


**Expected Output:**
URI for 47, XXY.

#### Case insensitive

We may use the `-i` option to perform a case insensitive matching. To check how many labels are now being recognized we can execute:

In [90]:
%%bash
grep -o -w -F -i -f diseases.txt chebi_27732_sentences.txt | sort -u | wc -l

57


**Expected Output:**
Count of case-insensitive matches.

To check which new labels were recognized, we can compare the results with and without the `-i` option:

In [91]:
%%bash
grep -o -w -F -i -f diseases.txt chebi_27732_sentences.txt | sort -u > diseases_recognized_ignorecase.txt

**Expected Output:**
No output (file created).

## Entity Linking

When we are using a generic lexicon, we may be interested in identifying what the recognized labels represent. For example, we may not be aware of what the matched label AD2 represents.

To solve this issue, we can use our script `geturi.sh` to perform entity linking (aka entity disambiguation, entity mapping, normalization), i.e. find the classes in the disease ontology that may be represented by the recognized label. For example, to find what AD2 represents, we can execute the following command:

In [92]:
%%bash
echo 'AD2' | ./geturi.sh doid.owl

http://purl.obolibrary.org/obo/DOID_0110035


**Expected Output:**
URI for AD2.

Only one URI is displayed.

Now we can retrieve other labels:

In [93]:
%%bash
echo 'http://purl.obolibrary.org/obo/DOID_0110035' | ./getlabels.sh doid.owl

AD2
Alzheimer disease 2, late onset
Alzheimer disease associated with APOE4
Alzheimer disease-2
Alzheimer's disease 2


**Expected Output:**
Alzheimer disease (and potentially synonyms).

In this case, the result clearly shows that AD2 represents the _Alzheimer disease_.

#### Modified labels

However, we may not be so lucky with the labels that were modified by our previous fixes in the lexicon. For example, we can test the case of ATR:

In [94]:
%%bash
echo 'ATR' | ./geturi.sh doid.owl

XPath set is empty


**Expected Output:**
XPath set is empty (or no match).

As expected, we received the warning that no URI was found.

```text
XPath set is empty
```

An approach to address this issue may involve keeping a track of the original label in a lexicon using another file.

#### Ambiguity

We may also have to deal with ambiguity problems where a label may represent multiple terms. For example, if we check how many classes the acronym KOS may represent:

In [95]:
%%bash
echo 'KOS' | ./geturi.sh doid.owl

http://purl.obolibrary.org/obo/DOID_0111456
http://purl.obolibrary.org/obo/DOID_0111712


**Expected Output:**
Two URIs.

We can see that it may represent two classes.

These two classes represent two distinct diseases, namely Kaufman oculocerebrofacial syndrome (DOID:0111456) and Kagami-Ogata syndrome (DOID:0111712), respectively.

We can also obtain their alternative labels by providing the two URI as standard input to the `getlabels.sh` script:

In [96]:
%%bash
echo 'http://purl.obolibrary.org/obo/DOID_0111456' | ./getlabels.sh doid.owl

KOS
blepharophimosis ptosis intellectual disability syndrome
oculocerebrofacial syndrome, Kaufman type
Kaufman oculocerebrofacial syndrome


**Expected Output:**
Labels for Kaufman oculocerebrofacial syndrome (including KOS).

In [97]:
%%bash
echo 'http://purl.obolibrary.org/obo/DOID_0111712' | ./getlabels.sh doid.owl

KOS
Kagami-Ogata syndrome


**Expected Output:**
Labels for Kagami-Ogata syndrome (including KOS).

We will get the following two lists, both containing KOS as expected.

If we find a `KOS` mention in the text, the challenge is to identify which of the syndromes the mention refers to. For addressing this challenge, we may have to use advanced entity linking techniques that analyze the context of the text.

#### Surrounding entities

An intuitive solution is to select the class closer in terms of meaning to the other classes mentioned in the surrounding text. This assumes that entities present in a piece of text are somehow semantically related to each other, which is normally the case. At least the author assumed some type of relation between them, otherwise the entities would not be in the same sentence.

Let us consider the following sentence about KOS:

```text
KOS is a syndromic intellectual disability
```

To identify the diseases in the previous sentence, we can execute the following command:

In [98]:
%%bash
echo 'KOS is a syndromic intellectual disability' | grep -o -w -F -f diseases.txt

KOS
syndromic intellectual disability


**Expected Output:**
Recognized diseases in the sentence.

We have a list of labels that can help us decide which is the right class representing KOS.

To find their URIs we can use the `geturi.sh` script:

In [99]:
%%bash
echo 'KOS is a syndromic intellectual disability' | grep -o -w -F -f diseases.txt | ./geturi.sh doid.owl

http://purl.obolibrary.org/obo/DOID_0111456
http://purl.obolibrary.org/obo/DOID_0111712
http://purl.obolibrary.org/obo/DOID_0050888


**Expected Output:**
URIs corresponding to the recognized diseases.

The only ambiguity is for KOS that returns two URIs, one representing the _Kaufman oculocerebrofacial syndrome _(DOID:0111456) and the other representing the _Kagami-Ogata syndrome_ (DOID:0111712).
The other URI represents the _Syndromic intellectual disability_ (DOID:0050888).

To decide which of the two URIs we should select, we can measure how close in meaning they are to the other diseases also found in the text.

#### Semantic similarity

Semantic similarity measures have been successfully applied to solve these ambiguity problems. Semantic similarity quantifies how close two classes are in terms of semantics encoded in a given ontology. Using the web tool Semantic Similarity Measures using Disjunctive Shared Information ([DiShIn](http://labs.rd.ciencias.ulisboa.pt/dishin/)), we can calculate the semantic similarity between our recognized classes. For example, we can calculate the similarity between _Kaufman oculocerebrofacial syndrome_ (DOID:0111456) and _Syndromic intellectual disability_ (DOID:0050888), and the similarity between _Kagami-Ogata syndrome_ (DOID:0111712) and _Syndromic intellectual disability_ (DOID:0050888).

We would see that for all measures _Syndromic intellectual disability_ is much more similar to _Kaufman oculocerebrofacial syndrome_ than to _Kagami-Ogata syndrome_. This means that by using semantic similarity we can automatically identify _Kaufman oculocerebrofacial syndrome_ as the correct linked entity for the mention KOS in this text.

## Large lexicons

The online tool [MER](https://github.com/lasigeBioTM/MER) is based on a shell script, so it can be easily executed as a command line to efficiently recognize and link entities using large lexicons.

#### MER installation



If we want to run MER locally, we can download the latest compressed file (zip) version and extract its contents:


In [100]:
%%bash
curl -O -L https://github.com/lasigeBioTM/MER/archive/master.zip
unzip master.zip
mv MER-master MER

Archive:  master.zip
5e12405475913a60dadea61c75b247697c146b3b
   creating: MER-master/
  inflating: MER-master/.gitignore   
  inflating: MER-master/Dockerfile   
  inflating: MER-master/Dockerfile-LexiconsSimilarity  
  inflating: MER-master/NOTICE.txt   
  inflating: MER-master/README.md    
  inflating: MER-master/benchmark.py  
   creating: MER-master/data/
  inflating: MER-master/data/lexicon.txt  
  inflating: MER-master/data/lexicon_links.tsv  
  inflating: MER-master/get_entities.sh  
  inflating: MER-master/get_similarity.sh  
  inflating: MER-master/produce_data_files.sh  
  inflating: MER-master/stopwords.txt  
  inflating: MER-master/test.sh      


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 19801    0 19801    0     0  23324      0 --:--:-- --:--:-- --:--:-- 23324


We now have to copy the Human Disease Ontology in to the data folder of MER:

In [101]:
%%bash
cp doid.owl MER/data/

**Expected Output:**
No output.

Lexicon files

To execute MER, we need first to create the lexicon files:

In [102]:
%%bash
(cd MER/data; ../produce_data_files.sh doid.owl)

myopathy
nanbh
neurocysticercosis
ophthalmoplegia
pigl.cdg
scoliosis
syndrome
zoophilia
zoophobia
zygomycosis
steinert disease
total ophthalmoplegia
westphal disease
xxy syndrome
xxy trisomy
zlotogora.ogur syndrome
zlotogora.zilberman.tenenbaum syndrome
zollinger.ellison syndrome
zunich.kaye syndrome
zygodactyly 1
syndactyly.ectodermal dysplasia.cleft.lip palate
syndromic intellectual disability
tapeworm infection. intestinal taenia solum
tapeworm infection. pork
tenia solium infectious disease
ventricular fibrillation with prolonged qt interval
viral hepatitis c
x.linked alport syndrome
ziziphus mauritiana fruit allergy
zunich neuroectodermal syndrome
stage i
syndactyly.ectodermal dysplasia.cleft.lip
syndromic intellectual
tapeworm infection.
tenia solium
ventricular fibrillation
viral hepatitis
x.linked alport
ziziphus mauritiana
zunich neuroectodermal


**Expected Output:**
Output logs from the production script.

This may take a few minutes to run. However, we only need to execute it once, each time we want to use a new version of the ontology. If we wait, the output will include the last patterns of each of the lexicon files.
We can check the contents of the created lexicons by using the `tail` command:

In [103]:
%%bash
tail MER/data/doid_*

==> MER/data/doid_links.tsv <==
ziziphus mauritiana fruit allergy	http://purl.obolibrary.org/obo/DOID_0060507
zlotogora-ogur syndrome	http://purl.obolibrary.org/obo/DOID_0080400
zlotogora-zilberman-tenenbaum syndrome	http://purl.obolibrary.org/obo/DOID_0060773
zollinger-ellison syndrome	http://purl.obolibrary.org/obo/DOID_0050782
zoophilia	http://purl.obolibrary.org/obo/DOID_9336
zoophobia	http://purl.obolibrary.org/obo/DOID_600
zunich-kaye syndrome	http://purl.obolibrary.org/obo/DOID_0112152
zunich neuroectodermal syndrome	http://purl.obolibrary.org/obo/DOID_0112152
zygodactyly 1	http://purl.obolibrary.org/obo/DOID_0111820
zygomycosis	http://purl.obolibrary.org/obo/DOID_8485

==> MER/data/doid_word1.txt <==
myopathy
nanbh
neurocysticercosis
ophthalmoplegia
pigl.cdg
scoliosis
syndrome
zoophilia
zoophobia
zygomycosis

==> MER/data/doid_word2.txt <==
steinert disease
total ophthalmoplegia
westphal disease
xxy syndrome
xxy trisomy
zlotogora.ogur syndrome
zlotogora.zilberman.tenenbaum synd

**Expected Output:**
Tail end of the generated data files.

These patterns are created according to the number of words of each term.

#### MER execution

Now we are ready to execute MER, by providing each sentence from the file `chebi_27732_sentences.txt` as argument to its `get_entities.sh` script.

In [104]:
%%bash
cd MER
cat ../chebi_27732_sentences.txt | tr -d "'" | xargs -I {} ./get_entities.sh '{}' doid

89	111	malignant hyperthermia	http://purl.obolibrary.org/obo/DOID_8545
74	96	malignant hyperthermia	http://purl.obolibrary.org/obo/DOID_8545
144	164	central core disease	http://purl.obolibrary.org/obo/DOID_3529
157	164	disease	http://purl.obolibrary.org/obo/DOID_4
13	20	disease	http://purl.obolibrary.org/obo/DOID_4
47	55	myopathy	http://purl.obolibrary.org/obo/DOID_423
254	274	central core disease	http://purl.obolibrary.org/obo/DOID_3529
267	274	disease	http://purl.obolibrary.org/obo/DOID_4
38	58	central core disease	http://purl.obolibrary.org/obo/DOID_3529
51	58	disease	http://purl.obolibrary.org/obo/DOID_4
68	87	congenital myopathy	http://purl.obolibrary.org/obo/DOID_0080100
79	87	myopathy	http://purl.obolibrary.org/obo/DOID_423
23	30	disease	http://purl.obolibrary.org/obo/DOID_4
166	173	disease	http://purl.obolibrary.org/obo/DOID_4
48	70	malignant hyperthermia	http://purl.obolibrary.org/obo/DOID_8545
59	81	malignant hyperthermia	http://purl.obolibrary.org/obo/DOID_8545
82	104	malign

**Expected Output:**
Entities recognized by MER with positions and URIs.

We removed single quotes from the text, since they are special characters to the command line `xargs`. Note that this is the get_entities.sh script inside the MER folder, not the one we created before. Now we will be able to obtain a large number of matches.

The first two numbers represent the start and end position of the match in the sentence. They are followed by the name of the disease and its URI in the ontology.

We can also redirect the output to a TSV file named `diseases_recognized.tsv`:

In [105]:
%%bash
cd MER
cat ../chebi_27732_sentences.txt | tr -d "'" | xargs -I {} ./get_entities.sh '{}' doid > ../diseases_recognized.tsv

**Expected Output:**
No output (file created).

We can now open the file in our spreadsheet application, such as LibreOffice Calc or Microsoft Excel.

## Conclusion



This concludes the **Unix Shell** tutorial adapted from the same section of the [Data and Text Processing for Health and Life Sciences](https://labs.rd.ciencias.ulisboa.pt/book/) book.

In this tutorial, we learned how to use XML parsing techniques to explore the semantics encoded in biomedical ontologies, which can be used to create lexicons that can then be used to recognize relevant entities in text.

This concludes the tutorial series. Feedback, typo reports, and suggestions for future additions are always welcome.

## Exercises


As an exercise, create a lexicon with all the names of compounds in ChEBI, and try to recognize them in the `chebi_27732_sentences.txt` file using MER.